# Cloud motion from brightness temperature (optical flow)

In [ ]:
%load_ext mcidasv_jupyter
%mcv_connect /path/to/runMcV
%mcv_replay off

## 1. Extract two consecutive temperature fields

In [ ]:
import mcidasv_jupyter as mcv
import numpy as np
session = mcv.get_session()

fields = session.extract_fields('''
adde = dict(server='adde.ucar.edu', dataset='EAST', descriptor='CONUSC13',
            size='ALL', unit='TEMP', mag=(-8, -8))
frames = [loadADDEImage(position=p, **adde) for p in (-1, 0)]
panel = buildWindow(height=400, width=500)
layer = panel[0].createLayer('Image Sequence Display', frames)
''', times=(0, 1))

f0 = np.nan_to_num(fields[0].masked(), nan=300.0)
f1 = np.nan_to_num(fields[1].masked(), nan=300.0)
nav = fields[0]
print(f0.shape, nav.unit)

## 2. Optical flow

In [ ]:
from skimage.registration import optical_flow_tvl1

v, u = optical_flow_tvl1(f0, f1)
speed_px = np.hypot(u, v)
print('max displacement: %.1f px/step' % speed_px.max())

## 3. Convert pixel motion to km/h using the navigation

In [ ]:
dlat_dy = np.gradient(nav.lats, axis=0)
dlon_dx = np.gradient(nav.lons, axis=1)
km_per_row = np.abs(dlat_dy) * 111.0
km_per_col = np.abs(dlon_dx) * 111.0 * np.cos(np.radians(nav.lats))
km = np.hypot(v * km_per_row, u * km_per_col)
speed = km * 4.0   # ~15-minute steps -> km/h
print('median cloud speed: %.0f km/h' % np.nanmedian(speed[np.isfinite(speed)]))

## 4. Plot flow field

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].imshow(f1, cmap='inferno_r'); ax[0].set_title('Tb (K)'); ax[0].axis('off')
st = max(1, max(speed.shape) // 40)
Y, X = np.mgrid[0:speed.shape[0]:st, 0:speed.shape[1]:st]
im = ax[1].imshow(speed, cmap='viridis'); ax[1].axis('off')
ax[1].quiver(X, Y, u[::st, ::st], -v[::st, ::st], color='white', scale=40)
ax[1].set_title('cloud motion (km/h)'); fig.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout()

## 5. Display the speed field in McIDAS-V

In [ ]:
from scipy.interpolate import griddata

def regrid(field, values, nlat=180, nlon=300, fill=np.nan):
    m = field.valid & np.isfinite(values)
    pts = np.column_stack([field.lats[m], field.lons[m]])
    glats = np.linspace(np.nanmax(field.lats[m]), np.nanmin(field.lats[m]), nlat)
    glons = np.linspace(np.nanmin(field.lons[m]), np.nanmax(field.lons[m]), nlon)
    GLA, GLO = np.meshgrid(glats, glons, indexing='ij')
    g = griddata(pts, values[m], (GLA, GLO), method='linear')
    return np.where(np.isfinite(g), g, fill).astype('f4'), glats, glons

grid, glats, glons = regrid(nav, speed, fill=0.0)
session.run('''
panel = buildWindow(height=500, width=750)
layer = panel[0].createLayer('Color-Shaded Plan View', g)
panel[0].setProjection('US>CONUS')
panel[0].setWireframe(False)
layer.setLayerLabel(label='cloud motion speed (km/h)')
''', arrays={'g': (grid, glats, glons)})